In [ ]:
import config
import tensorflow.keras as keras
import tensorflow as tf
from CustomLossFunction import DiceLoss
from Generator import DataGenerator
import Generator
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ModelBuilding import CreateModel
os.getcwd()

In [ ]:
# model = keras.models.load_model(os.path.sep.join(["output","detector20201216-181001.h5"]), compile=False)   
model = CreateModel()
# keras.models.load_model(os.path.sep.join(["output","models","bests", "model.01-6.70.h5"]), compile=True, custom_objects={'DiceLoss':DiceLoss})

In [ ]:
losses = {
        "class_label": tf.keras.losses.categorical_crossentropy,
        "bounding_box": DiceLoss,
    }
model.load_weights(os.path.sep.join(["output","models","bests", "model.02--13.46.h5"]))
model.compile(loss=losses)

In [ ]:
datagen = DataGenerator(['kidneyHU.png', 'lungHU.png'], '.', to_fit=False, csvpath=config.ANNOTS_PATH)

In [ ]:
imgarrs = [Generator.ImagePreprocessing(i, (512,512)) for i in ['kidneyHU.png', 'lungHU.png']]
imgarrs = np.array(imgarrs)
print(imgarrs.shape)

In [ ]:
output = model(imgarrs, training=False)

In [ ]:
for i in range(len(output)):
    print(output[i].shape)

In [ ]:
for i in range(2):
    leison_types = output[0][i]
    aimg = output[1][i]
#     break
    num = np.argmax(leison_types)
    
    print(num, config.NUM_TYPE_MAPPING[str(num+1)])

In [ ]:
# mask = generator.ImagePreprocessing('kidneyHU.png', (512,512))
mask = datagen._generate_y(['000010_01_01_084.png'])['bounding_box']
# mask = np.delete(mask, np.s_[1:], axis=-1).reshape(512,512)
# print(mask)
original = plt.imshow(mask.reshape(512,512))
plt.colorbar()

In [ ]:
numpymask = np.array(output[1][0]).squeeze()
# plt.hist(numpymask.ravel(), bins=256, range=(0.0, 1.0), fc='k', ec='k')
imgplot = plt.imshow(numpymask)
plt.colorbar()

In [ ]:
numpymask = np.array(np.array(output[1][0]) == 0, dtype=int)
numpymask.resize(512,512)
print(numpymask.shape)
# plt.imshow(numpymask)
plt.imshow([[1,1], [0,0]])

In [ ]:
contours,_ = cv2.findContours(numpymask.copy(), 1, 1) # not copying here will throw an error
rect = cv2.minAreaRect(contours[0]) # basically you can feed this rect into your classifier
(x,y),(w,h), a = rect # a - angle

box = cv2.boxPoints(rect)
box = np.int0(box) #turn into ints
rect2 = cv2.drawContours(img.copy(),[box],0,(0,0,255),10)

plt.imshow(rect2)
plt.show()